# 01: Data Ingestion

Use this notebook to download and inspect GTFS, census, LEHD, and OSM data.

Suggested steps:
- Load GTFS zip files from `data_raw/`
- Download census tract shapefile and ACS tables (default 2018–2022 5-year ACS)
- Download LEHD (LODES) jobs data
- Try OSMnx to fetch the walking network (catch exceptions and retry if needed)
- Save any clean outputs to `outputs/`



In [10]:
from pathlib import Path
import os
import numpy as np

ROOT = Path("/Users/aaryakhanna/transit-deserts").resolve()
os.chdir(ROOT)
print("Working directory set to:", Path().resolve())

data_raw = ROOT / "data_raw"
data_raw.mkdir(exist_ok=True)
print("data_raw folder exists:", data_raw.exists())

outputs = ROOT / "outputs"
outputs.mkdir(exist_ok=True)
print("outputs folder exists:", outputs.exists())


Working directory set to: /Users/aaryakhanna/transit-deserts
data_raw folder exists: True
outputs folder exists: True


In [11]:
for path in sorted(data_raw.iterdir()):
    print(path.name)



acs_2022_la_county.csv
ca_wac_S000_JT00_2021.csv.gz
metro_bus_gtfs.zip
metro_rail_gtfs.zip
tl_2025_06_tract.cpg
tl_2025_06_tract.dbf
tl_2025_06_tract.prj
tl_2025_06_tract.shp
tl_2025_06_tract.shp.ea.iso.xml
tl_2025_06_tract.shp.iso.xml
tl_2025_06_tract.shx
tl_2025_06_tract.zip


In [12]:
import geopandas as gpd
import gtfs_kit as gk
import osmnx as ox
from zipfile import ZipFile

In [13]:
import requests, pandas as pd

API_KEY = "6debb1fa0e72f7047965b9c1726f92d4ec8a57de"
base_url = "https://api.census.gov/data/2022/acs/acs5"  # 2018–2022 ACS 5-year release
variables = [
   "B01003_001E",  # total population
   "B19013_001E",  # median household income
   "B25046_001E",  # aggregate vehicles available
   "B08126_001E",  # commute by poverty status (transit)
   "B08301_001E"   # overall commute mode split
]
params = {
   "get": "NAME," + ",".join(variables),
   "for": "tract:*",
   "in": "state:06 county:037",  # 06=CA, 037=Los Angeles County
   "key": API_KEY
}
response = requests.get(base_url, params=params)
response.raise_for_status()
data = response.json()
df = pd.DataFrame(data[1:], columns=data[0])
output_path = data_raw / "acs_2022_la_county.csv"
df.to_csv(output_path, index=False)
print(f"Saved ACS table to {output_path.relative_to(ROOT)} with {len(df)} tracts")

Saved ACS table to data_raw/acs_2022_la_county.csv with 2498 tracts


In [14]:
acs_path = data_raw / "acs_2022_la_county.csv"
acs = pd.read_csv(acs_path)
print(acs.head())
print(acs.columns)
print("Rows:", len(acs))


                                                NAME  B01003_001E  \
0  Census Tract 1011.10; Los Angeles County; Cali...         4014   
1  Census Tract 1011.22; Los Angeles County; Cali...         4164   
2  Census Tract 1012.20; Los Angeles County; Cali...         3481   
3  Census Tract 1012.21; Los Angeles County; Cali...         3756   
4  Census Tract 1012.22; Los Angeles County; Cali...         2808   

   B19013_001E  B25046_001E  B08126_001E  B08301_001E  state  county   tract  
0        68972       2681.0         2043         2043      6      37  101110  
1       118859       3152.0         2014         2014      6      37  101122  
2        65139       2599.0         1665         1665      6      37  101220  
3        53348       2365.0         1762         1762      6      37  101221  
4        36779          NaN         1105         1105      6      37  101222  
Index(['NAME', 'B01003_001E', 'B19013_001E', 'B25046_001E', 'B08126_001E',
       'B08301_001E', 'state', 'coun

In [15]:
tracts_path = data_raw / "tl_2025_06_tract.shp"
tracts = gpd.read_file(tracts_path)
print(tracts.head())
print(tracts.columns)
print("Total tracts statewide:", len(tracts))

tracts_la = tracts[tracts['COUNTYFP'] == '037'].copy()
print("LA County tracts:", len(tracts_la))


  STATEFP COUNTYFP TRACTCE        GEOID               GEOIDFQ    NAME  \
0      06      065  042516  06065042516  1400000US06065042516  425.16   
1      06      065  042716  06065042716  1400000US06065042716  427.16   
2      06      065  042717  06065042717  1400000US06065042717  427.17   
3      06      065  042902  06065042902  1400000US06065042902  429.02   
4      06      065  042903  06065042903  1400000US06065042903  429.03   

              NAMELSAD  MTFCC FUNCSTAT     ALAND  AWATER     INTPTLAT  \
0  Census Tract 425.16  G5020        S    973130       0  +33.9227363   
1  Census Tract 427.16  G5020        S   2698784  711913  +33.6865062   
2  Census Tract 427.17  G5020        S   6425166  145210  +33.6997108   
3  Census Tract 429.02  G5020        S  43393952       0  +33.7493422   
4  Census Tract 429.03  G5020        S  44642454       0  +33.8027798   

       INTPTLON                                           geometry  
0  -117.2393879  POLYGON ((-117.24375 33.92818, -117.

In [16]:
# w_geocode format: 15 digits = state(2) + county(3) + tract(6) + block(4)
# IMPORTANT: Read w_geocode as string to preserve leading zeros
# LA County FIPS = 037, so positions 2-5 should be "037" (after zero-padding to 15 digits)
lehd_path = data_raw / "ca_wac_S000_JT00_2021.csv.gz"
lehd_cols = ["w_geocode", "C000"]  # total jobs per workplace block

print("Loading LEHD data (this may take 30-60 seconds)...")
# CRITICAL: Read w_geocode as string dtype to preserve leading zeros
chunks = []
chunk_count = 0
for chunk in pd.read_csv(lehd_path, usecols=lehd_cols, chunksize=50000, dtype={'w_geocode': str}):
    chunk_count += 1
    # Zero-pad to 15 digits, then check county code (positions 2-5)
    chunk_str = chunk['w_geocode'].str.zfill(15)
    # LA County FIPS is 037, check positions 2-5
    la_mask = chunk_str.str[2:5] == '037'
    if la_mask.any():
        chunks.append(chunk[la_mask])
    if chunk_count % 10 == 0:
        print(f"  Processed {chunk_count} chunks, found {len(chunks)} chunks with LA County data")

if chunks:
    lehd = pd.concat(chunks, ignore_index=True)
    print(f"✓ Loaded {len(lehd):,} LA County workplace blocks")
    print(lehd.head())
else:
    print("⚠ No LA County data found in chunks - trying full file read")
    lehd_full = pd.read_csv(lehd_path, usecols=lehd_cols, dtype={'w_geocode': str})
    lehd_str = lehd_full['w_geocode'].str.zfill(15)
    la_mask = lehd_str.str[2:5] == '037'
    lehd = lehd_full[la_mask].copy()
    print(f"✓ Loaded {len(lehd):,} LA County workplace blocks from full file")


Loading LEHD data (this may take 30-60 seconds)...
✓ Loaded 62,924 LA County workplace blocks
         w_geocode  C000
0  060371011101000     4
1  060371011101001     3
2  060371011101002     5
3  060371011101003     5
4  060371011101004     9


In [17]:
# First 11 digits = state (2) + county (3) + tract (6) = tract GEOID
# w_geocode is already string, just zero-pad to 15 and slice first 11
print("Aggregating jobs to tract level...")
lehd['tract_geoid'] = lehd['w_geocode'].str.zfill(15).str[:11]
jobs_by_tract_la = lehd.groupby('tract_geoid', as_index=False)['C000'].sum()
jobs_by_tract_la = jobs_by_tract_la.rename(columns={'C000': 'jobs_total'})
jobs_by_tract_la['tract_geoid'] = jobs_by_tract_la['tract_geoid'].astype(str)
print(jobs_by_tract_la.head())
print(f"✓ {len(jobs_by_tract_la):,} LA County tracts with job data")
print(f"Sample GEOID format: {jobs_by_tract_la['tract_geoid'].iloc[0]} (should be 11 digits starting with 06037)")
print(f"All start with 06037: {jobs_by_tract_la['tract_geoid'].str.startswith('06037').all()}")


Aggregating jobs to tract level...
   tract_geoid  jobs_total
0  06037101110         634
1  06037101122         279
2  06037101220         422
3  06037101221         454
4  06037101222         561
✓ 2,494 LA County tracts with job data
Sample GEOID format: 06037101110 (should be 11 digits starting with 06037)
All start with 06037: True


In [18]:
acs = acs.rename(columns={
    "B01003_001E": "pop_total",
    "B19013_001E": "median_income",
    "B25046_001E": "vehicles_total",
    "B08126_001E": "transit_commuters_poverty",
    "B08301_001E": "commute_total"
})

# IMPORTANT: Census API uses -666666666 as missing data code for income
acs['median_income'] = acs['median_income'].replace(-666666666, np.nan)

# Format GEOID: state (2 digits) + county (3 digits) + tract (6 digits) = 11 digits
acs['state_padded'] = acs['state'].astype(str).str.zfill(2)
acs['county_padded'] = acs['county'].astype(str).str.zfill(3)
acs['tract_padded'] = acs['tract'].astype(str).str.zfill(6)
acs['tract_geoid'] = acs['state_padded'] + acs['county_padded'] + acs['tract_padded']
acs_la = acs[acs['county'].astype(str) == '37'].copy()
print(acs_la[['tract_geoid','pop_total','median_income']].head())
print("ACS tracts in LA County:", len(acs_la))
print(f"Tracts with valid income: {acs_la['median_income'].notna().sum()}")


   tract_geoid  pop_total  median_income
0  06037101110       4014        68972.0
1  06037101122       4164       118859.0
2  06037101220       3481        65139.0
3  06037101221       3756        53348.0
4  06037101222       2808        36779.0
ACS tracts in LA County: 2498
Tracts with valid income: 2449


In [19]:
acs_cols_to_merge = ['tract_geoid', 'pop_total', 'median_income', 'vehicles_total', 
                     'transit_commuters_poverty', 'commute_total']
tracts_la = tracts_la.merge(acs_la[acs_cols_to_merge], left_on='GEOID', right_on='tract_geoid', how='left')
tracts_la = tracts_la.merge(jobs_by_tract_la, left_on='GEOID', right_on='tract_geoid', how='left', suffixes=('', '_jobs'))
tracts_la = tracts_la.drop(columns=['tract_geoid', 'tract_geoid_jobs'], errors='ignore')
print(tracts_la[['GEOID','pop_total','median_income','jobs_total']].head())
print("Final tract rows:", len(tracts_la))
print("Tracts with population data:", tracts_la['pop_total'].notna().sum())
print("Tracts with jobs data:", tracts_la['jobs_total'].notna().sum())


         GEOID  pop_total  median_income  jobs_total
0  06037432300       3922        73250.0      2008.0
1  06037432401       3951        75000.0       717.0
2  06037432402       5480        56732.0      1457.0
3  06037432601       6489        69214.0       702.0
4  06037461400       2736        78594.0       678.0
Final tract rows: 2498
Tracts with population data: 2498
Tracts with jobs data: 2494


In [20]:
print("Loading GTFS feeds...")
stops_list = []
for label in ["bus", "rail"]:
    gtfs_zip = data_raw / f"metro_{label}_gtfs.zip"
    if gtfs_zip.exists():
        print(f"  Reading {gtfs_zip.name}...")
        feed = gk.read_feed(gtfs_zip, dist_units="km")
        stops = gk.get_stops(feed)
        gdf = gpd.GeoDataFrame(
            stops,
            geometry=gpd.points_from_xy(stops.stop_lon, stops.stop_lat),
            crs="EPSG:4326"
        )
        gdf['mode'] = label
        stops_list.append(gdf)
        print(f"    ✓ {len(gdf)} stops")
    else:
        print(f"  Missing {gtfs_zip.name}")

if stops_list:
    stops_all = gpd.GeoDataFrame(pd.concat(stops_list, ignore_index=True), crs="EPSG:4326")
    print(f"✓ Total stops: {len(stops_all):,}")
    print(stops_all[['stop_name', 'mode']].head())


Loading GTFS feeds...
  Reading metro_bus_gtfs.zip...


/Users/aaryakhanna/transit-deserts/.venv/lib/python3.13/site-packages/gtfs_kit/feed.py:392: DtypeWarning: Columns (10,12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


    ✓ 11838 stops
  Reading metro_rail_gtfs.zip...
    ✓ 440 stops
✓ Total stops: 12,278
                             stop_name mode
0                  Paramount / Slauson  bus
1                     Jefferson / 10th  bus
2           120th / Augustus F Hawkins  bus
3  120th / Martin Luther King Hospital  bus
4                    15054 Sherman Way  bus


/var/folders/jp/c8jhgklj2l71r2jgjptsxm6m0000gn/T/ipykernel_7088/2169797246.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  stops_all = gpd.GeoDataFrame(pd.concat(stops_list, ignore_index=True), crs="EPSG:4326")


In [21]:
print("\nSaving outputs...")
import pyarrow.parquet as pq

tracts_out = outputs / "la_tracts_with_acs_jobs.parquet"
tracts_la.to_parquet(tracts_out, index=False)
print(f"✓ Saved {tracts_out.relative_to(ROOT)} ({len(tracts_la):,} tracts)")

tracts_geojson = outputs / "la_tracts_with_acs_jobs.geojson"
tracts_la.to_file(tracts_geojson, driver="GeoJSON")
print(f"✓ Saved {tracts_geojson.relative_to(ROOT)}")

stops_out = outputs / "metro_stops_all.geojson"
if 'stops_all' in locals():
    stops_all.to_file(stops_out, driver="GeoJSON")
    print(f"✓ Saved {stops_out.relative_to(ROOT)} ({len(stops_all):,} stops)")

jobs_out = outputs / "jobs_by_tract_la.csv"
jobs_by_tract_la.to_csv(jobs_out, index=False)
print(f"✓ Saved {jobs_out.relative_to(ROOT)} ({len(jobs_by_tract_la):,} tracts)")
print("\n✅ Data ingestion complete!")



Saving outputs...
✓ Saved outputs/la_tracts_with_acs_jobs.parquet (2,498 tracts)
✓ Saved outputs/la_tracts_with_acs_jobs.geojson
✓ Saved outputs/metro_stops_all.geojson (12,278 stops)
✓ Saved outputs/jobs_by_tract_la.csv (2,494 tracts)

✅ Data ingestion complete!


In [22]:
# Optional: download OSM walking network (SKIP for now - can do later)
# This can take 10+ minutes and may timeout. We'll build network in notebook 02.
print("Skipping OSM download for now.")
print("We'll build the walking network in notebook 02 when needed.")
print("If you want to try it manually, use:")
print("  G = ox.graph_from_bbox(bbox=(33.7, 34.34, -118.7, -118.0), network_type='walk')")

Skipping OSM download for now.
We'll build the walking network in notebook 02 when needed.
If you want to try it manually, use:
  G = ox.graph_from_bbox(bbox=(33.7, 34.34, -118.7, -118.0), network_type='walk')
